# ReWOO 和 规划-执行，接耦计划

ReAct 把思考和行动交错在同一个流里。ReWOO将它们分离：一个大的计划先行，然后执行。Plan-and-Execute是它的泛化；Plan-and-Act进一步拓展到页面导航。


## 问题描述

ReAct 交错的思考-行动-观察训练简单且灵活，但是每个工具调用都需要带着之前所有的上下文，包括以前每步的思考。Token 用量随深度二次增长。更糟糕的是，当循环过程中出现了一个工作错误，模型需要根据错误的观察结果重新规划整个计划。

ReWOO 下了个赌注：先把所有的事情计划好，并行的获取争取，最后再组装答案。一次LLM调用生成计划，N次工具调用获取证据（可并行），一次LLM调用最后解决问题。

## 基本概念

### 三个角色

- Planner。  从用户问题出计划DAG。
- Workders。 按计划DAG执行获取证据。
- Solver。   从用户问题，计划DAG以及证据生成最终答案。

Planer生成一张DAG，每个节点是一个工具调用，包括入参、以及它依赖的节点。Workders 根据拓扑顺序执行节点。Solver最后将所有的东西缝到一起。

### 问什么Token消耗少5倍

ReAct 提示词随步数线形增长。在第10步的时候，提示词中包含思考1、行动1、观察1、思考2、行动2、观察2... 每个中间步也会被冗余地包含在原始提示词中。

ReWOO 只付出一次planner prompt（大），然后是N次worker prompts（小，只有工具调用，没有链），以及一次solver prompt。

### 为什么更鲁棒

ReAct 中如果worker3 失败了，循环需要在流中间针对错误进行推导。而在ReWOO 中，workder3 返回一个错误字符串，solver在带着原始计划的上下文中看见它，可以优雅降级。失败定位是按节点的，而不是按步骤的。

### Planner 蒸馏

因为planner不处理观察结果，所以你可以蒸馏一个模型来制定计划。2026年大多生产级agents 使用一个小的planner 加上一个大的executor，或者反过来。

### Plan-and-Execute

LangChain 对ReWOO 泛化后的模式名。先行的planer吐出一个步骤列表，executor 执行每个步骤，一个可选的replanner用于在观察结果后修订计划。比起ReWOO 更贴近ReAct（replanner 将观测结果带回到planning），但是保留了token节省。

### Plan-and-Act

Plan-and-Act 将这种模式推广到了长视野的web和移动设别agents。关键是训练数据带来的增强。Executor将高层计划落成环境动作（点击、输入、导航等），用合成轨迹给Planner（及Executor）做训练/微调。

### 怎么选

|模式|什么时候选|
|---|---|
|ReAct|短任务、环境未知、需要交互式的异常处理|
|ReWOO|基于已知工具的结构化任务、token敏感、并行举证|
|Plan-and-Execute|与ReWOO相同，不过支持replanning|
|Plan-and-Act|长视野（>30步），网页/移动设备/电脑使用|
|Tree of Thoughts|搜索值得为之买单|

Anthropic 的指导：从最简单的开始。如果任务只需要一次工具调用外加一次总结，不要构建ReWOO。如果任务是一个40步的研究作业，不要只使用ReAct。

# 开始编码

本章核心：**Planner → Workers（可并行）→ Solver**（ReWOO），以及带 **Replanner** 的 Plan-and-Execute。  
先用纯 Python 玩具跑通 DAG；再用 **PyTorch 蒸馏一个小 Planner**；最后用 **LangGraph + DeepSeek** 做生产示意。


## 1. 教学玩具：ReWOO（计划 DAG + 拓扑执行 + Solver）


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
import re
from typing import Any, Callable

from typing_extensions import TypedDict


@dataclass(frozen=True)
class PlanNode:
    """计划 DAG 中的一个工具调用节点。"""

    id: str
    tool: str
    args: dict[str, Any]
    depends_on: tuple[str, ...] = ()
    """必须先完成的上游节点 id。"""


@dataclass
class Evidence:
    """某个计划节点的执行结果（观察）。"""

    node_id: str
    tool: str
    ok: bool
    content: str


class ToolRegistry:
    """按名字分发工具；失败返回字符串，不抛出。"""

    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., str]] = {}

    def register(self, name: str, fn: Callable[..., str]) -> None:
        """
        Args:
            name: 工具名。
            fn: 返回 ``str`` 的可调用对象。
        """
        self._tools[name] = fn

    def names(self) -> list[str]:
        """
        Returns:
            names: 已注册工具名。
        """
        return sorted(self._tools)

    def dispatch(self, name: str, args: dict[str, Any]) -> str:
        """
        Args:
            name: 工具名。
            args: 关键字参数。

        Returns:
            result: 成功输出或 ``Error: ...``。
        """
        fn = self._tools.get(name)
        if fn is None:
            return f"Error: unknown tool '{name}'. Available={self.names()}"
        try:
            return fn(**args)
        except TypeError as e:
            return f"Error: bad args {args}: {e}"
        except Exception as e:
            return f"Error: {type(e).__name__}: {e}"


def calculator(expr: str) -> str:
    """
    Args:
        expr: 白名单算术表达式。

    Returns:
        value: 计算结果或错误信息。
    """
    allowed = set("0123456789+-*/(). ")
    if not set(expr).issubset(allowed):
        return "Error: illegal characters"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))  # noqa: S307
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"


def search_kb(query: str) -> str:
    """
    玩具知识库检索。

    Args:
        query: 查询字符串。

    Returns:
        snippet: 命中片段或未找到。
    """
    kb = {
        "capital of france": "Paris",
        "pi approx": "22/7",
        "answer key": "use calculator for arithmetic",
    }
    q = query.lower().strip()
    for k, v in kb.items():
        if k in q or q in k:
            return v
    # 宽松匹配：同时含关键词
    if "france" in q and "capital" in q:
        return "Paris"
    return f"Error: missing kb entry for {query}"


def topological_levels(nodes: list[PlanNode]) -> list[list[PlanNode]]:
    """
    把 DAG 分层，同层节点可并行执行。

    Args:
        nodes: 计划节点列表。

    Returns:
        levels: 每一层是一批无相互依赖的节点。

    Raises:
        ValueError: 存在未知依赖或环。
    """
    by_id = {n.id: n for n in nodes}
    if len(by_id) != len(nodes):
        raise ValueError("duplicate node id")
    indeg = {n.id: 0 for n in nodes}
    children: dict[str, list[str]] = {n.id: [] for n in nodes}
    for n in nodes:
        for d in n.depends_on:
            if d not in by_id:
                raise ValueError(f"unknown dependency {d} for {n.id}")
            indeg[n.id] += 1
            children[d].append(n.id)

    ready = [i for i, c in indeg.items() if c == 0]
    levels: list[list[PlanNode]] = []
    seen = 0
    while ready:
        layer_ids = sorted(ready)
        ready = []
        levels.append([by_id[i] for i in layer_ids])
        seen += len(layer_ids)
        for i in layer_ids:
            for c in children[i]:
                indeg[c] -= 1
                if indeg[c] == 0:
                    ready.append(c)
    if seen != len(nodes):
        raise ValueError("cycle detected in plan DAG")
    return levels


def resolve_args(args: dict[str, Any], evidence: dict[str, Evidence]) -> dict[str, Any]:
    """
    支持参数中的 ``$node_id`` 占位（整值或内嵌于字符串）。

    Args:
        args: 原始参数。
        evidence: 已完成节点的证据表。

    Returns:
        resolved: 解析后的参数字典。
    """

    def _sub(text: str) -> str:
        def repl(m: re.Match[str]) -> str:
            ref = m.group(1)
            if ref not in evidence:
                return f"<missing:{ref}>"
            return evidence[ref].content

        return re.sub(r"\$([A-Za-z_][A-Za-z0-9_]*)", repl, text)

    out: dict[str, Any] = {}
    for k, v in args.items():
        if isinstance(v, str):
            out[k] = _sub(v)
        else:
            out[k] = v
    return out


def run_workers(
    nodes: list[PlanNode],
    tools: ToolRegistry,
) -> dict[str, Evidence]:
    """
    按拓扑层执行 Workers（同层示意并行：顺序调用但无交叉依赖）。

    Args:
        nodes: 计划 DAG。
        tools: 工具表。

    Returns:
        evidence: ``node_id → Evidence``。
    """
    evidence: dict[str, Evidence] = {}
    for level in topological_levels(nodes):
        for node in level:
            args = resolve_args(node.args, evidence)
            raw = tools.dispatch(node.tool, args)
            ok = not str(raw).startswith("Error:")
            evidence[node.id] = Evidence(node.id, node.tool, ok, str(raw))
    return evidence


def estimate_react_prompt_units(n_steps: int, base: int = 10, step_cost: int = 8) -> int:
    """
    粗估 ReAct 提示词“单位成本”（随步数近似二次累计）。

    Args:
        n_steps: 工具步数。
        base: 固定前缀成本。
        step_cost: 每步 thought+action+obs 增量。

    Returns:
        units: ``base + sum_{i=1..n} i*step_cost`` 的简化上界示意。
    """
    return base + step_cost * n_steps * (n_steps + 1) // 2


def estimate_rewoo_prompt_units(
    n_workers: int,
    planner: int = 40,
    worker: int = 6,
    solver: int = 30,
) -> int:
    """
    粗估 ReWOO：一次 planner + N 次小 worker + 一次 solver。

    Args:
        n_workers: worker 调用次数。
        planner: planner prompt 成本。
        worker: 单次 worker 成本。
        solver: solver 成本。

    Returns:
        units: 总和。
    """
    return planner + n_workers * worker + solver


@dataclass
class ReWOOResult:
    """一次 ReWOO 运行结果。"""

    question: str
    plan: list[PlanNode]
    evidence: dict[str, Evidence]
    answer: str
    react_units: int
    rewoo_units: int


def toy_solver(question: str, plan: list[PlanNode], evidence: dict[str, Evidence]) -> str:
    """
    玩具 Solver：把证据缝成最终答案（真实系统换 LLM）。

    Args:
        question: 用户问题。
        plan: 原计划。
        evidence: 节点证据。

    Returns:
        answer: 最终回答。
    """
    lines = [f"Q: {question}", "Plan + Evidence:"]
    for n in plan:
        ev = evidence[n.id]
        status = "ok" if ev.ok else "FAIL"
        lines.append(f"- {n.id}: {n.tool}({n.args}) -> [{status}] {ev.content}")
    fails = [e for e in evidence.values() if not e.ok]
    if fails:
        lines.append("Degraded: some workers failed; answer uses remaining evidence.")
    # 启发式：若有 calculator 结果则突出
    nums = [e.content for e in evidence.values() if e.ok and e.tool == "calculator"]
    if nums:
        lines.append(f"Answer: computed value(s) = {', '.join(nums)}")
    else:
        ok_bits = [e.content for e in evidence.values() if e.ok]
        lines.append("Answer: " + ("; ".join(ok_bits) if ok_bits else "insufficient evidence"))
    return "\n".join(lines)


def run_rewoo(
    question: str,
    plan: list[PlanNode],
    tools: ToolRegistry,
) -> ReWOOResult:
    """
    完整玩具 ReWOO：给定（脚本化）计划，执行 workers，再 solver。

    Args:
        question: 用户问题。
        plan: Planner 产出的 DAG。
        tools: 工具表。

    Returns:
        result: 含证据、答案与 token 成本对比粗估。
    """
    evidence = run_workers(plan, tools)
    answer = toy_solver(question, plan, evidence)
    return ReWOOResult(
        question=question,
        plan=plan,
        evidence=evidence,
        answer=answer,
        react_units=estimate_react_prompt_units(len(plan)),
        rewoo_units=estimate_rewoo_prompt_units(len(plan)),
    )


print("ReWOO toy ready | DAG workers + solver + cost estimators")


## 2. 玩具示例：并行层、失败降级、成本对比


In [ ]:
def build_default_tools() -> ToolRegistry:
    """
    Returns:
        registry: 含 ``calculator`` 与 ``search_kb``。
    """
    reg = ToolRegistry()
    reg.register("calculator", calculator)
    reg.register("search_kb", search_kb)
    return reg


def demo_rewoo_parallel_and_degrade() -> None:
    """两层 DAG：检索与无关计算可同层；下游依赖上游结果。"""
    tools = build_default_tools()
    question = "法国首都是什么？并把 3*(1+2) 算出来。"
    plan = [
        PlanNode("n1", "search_kb", {"query": "capital of france"}),
        PlanNode("n2", "calculator", {"expr": "3*(1+2)"}),  # 与 n1 同层可并行
        PlanNode(
            "n3",
            "calculator",
            {"expr": "10+$n2"},  # 依赖 n2 的数值；玩具 resolve 会得到非法式 → 演示失败降级
            depends_on=("n2",),
        ),
    ]
    # 修正 n3：玩具里 $ 替换成字符串 "6"，拼成 "10+6" 合法
    plan[2] = PlanNode("n3", "calculator", {"expr": "10+$n2"}, depends_on=("n2",))

    levels = topological_levels(plan)
    print("=== topological levels (parallel batches) ===")
    for i, layer in enumerate(levels):
        print(f"level {i}: {[n.id for n in layer]}")

    result = run_rewoo(question, plan, tools)
    print("\n=== evidence ===")
    for nid, ev in result.evidence.items():
        print(f"{nid}: ok={ev.ok} {ev.content}")
    print("\n=== solver ===")
    print(result.answer)
    # 小步数时 planner/solver 常数开销占优；长链时 ReAct 二次项更明显
    n = 12
    print(
        f"\ncost units (this run): ReAct≈{result.react_units} vs ReWOO≈{result.rewoo_units}"
    )
    print(
        f"cost units (n={n} hypothetical): "
        f"ReAct≈{estimate_react_prompt_units(n)} vs ReWOO≈{estimate_rewoo_prompt_units(n)} "
        f"(ratio≈{estimate_react_prompt_units(n) / estimate_rewoo_prompt_units(n):.2f}x)"
    )
    assert result.evidence["n1"].content == "Paris"
    assert result.evidence["n2"].content == "9"
    assert result.evidence["n3"].ok


def demo_worker_failure_localized() -> None:
    """故意让一个 worker 失败，Solver 仍能用其余证据降级作答。"""
    tools = build_default_tools()
    plan = [
        PlanNode("a", "search_kb", {"query": "capital of france"}),
        PlanNode("b", "search_kb", {"query": "this will miss"}),
    ]
    result = run_rewoo("首都?", plan, tools)
    print("\n=== localized failure ===")
    print(result.answer)
    assert result.evidence["a"].ok and not result.evidence["b"].ok
    assert "Paris" in result.answer


demo_rewoo_parallel_and_degrade()
demo_worker_failure_localized()
print("\nTOY DEMOS OK")


## 3. PyTorch：蒸馏小 Planner（给定问题 → 工具序列分数）

笔记要点：Planner 不吃 Observation，可单独蒸馏。这里用 bag-of-words + 线性层预测「该用哪些工具」，纯教学规模。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


TOOL_VOCAB: list[str] = ["calculator", "search_kb"]
WORD_VOCAB: list[str] = [
    "capital",
    "france",
    "compute",
    "calculate",
    "math",
    "search",
    "what",
    "is",
    "plus",
    "times",
]


def tokenize_question(text: str) -> torch.Tensor:
    """
    Args:
        text: 用户问题。

    Returns:
        bag: ``(V,)`` 词袋多热向量，``V=len(WORD_VOCAB)``。
    """
    tokens = set(text.lower().replace("?", " ").replace(",", " ").split())
    x = torch.zeros(len(WORD_VOCAB), dtype=torch.float32)
    for i, w in enumerate(WORD_VOCAB):
        if w in tokens:
            x[i] = 1.0
    return x


def tool_multihot(tools: list[str]) -> torch.Tensor:
    """
    Args:
        tools: 计划中应出现的工具名列表。

    Returns:
        y: ``(T,)`` 多标签目标，``T=len(TOOL_VOCAB)``。
    """
    y = torch.zeros(len(TOOL_VOCAB), dtype=torch.float32)
    for t in tools:
        if t in TOOL_VOCAB:
            y[TOOL_VOCAB.index(t)] = 1.0
    return y


class DistilledPlanner(nn.Module):
    """极小 Planner：词袋 → 工具多标签 logits（蒸馏占位）。"""

    def __init__(self, n_words: int = len(WORD_VOCAB), n_tools: int = len(TOOL_VOCAB)) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_words, 32),
            nn.ReLU(),
            nn.Linear(32, n_tools),
        )

    def forward(self, bag: torch.Tensor) -> torch.Tensor:
        """
        Args:
            bag: ``(B, V)`` 或 ``(V,)``。

        Returns:
            logits: ``(B, T)`` 或 ``(T,)``。
        """
        single = bag.ndim == 1
        if single:
            bag = bag.unsqueeze(0)
        logits = self.net(bag)
        return logits.squeeze(0) if single else logits

    @torch.no_grad()
    def predict_tools(self, question: str, threshold: float = 0.5) -> list[str]:
        """
        Args:
            question: 用户问题。
            threshold: sigmoid 阈值。

        Returns:
            tools: 预测需要的工具名。
        """
        logits = self.forward(tokenize_question(question))
        probs = torch.sigmoid(logits)
        return [TOOL_VOCAB[i] for i, p in enumerate(probs.tolist()) if p >= threshold]


def train_distilled_planner(
    steps: int = 200,
    lr: float = 0.05,
) -> DistilledPlanner:
    """
    在合成 (问题, 工具集合) 上 BCE 训练。

    Args:
        steps: 优化步数。
        lr: 学习率。

    Returns:
        model: 训练后的 ``DistilledPlanner``。
    """
    data: list[tuple[str, list[str]]] = [
        ("what is capital of france", ["search_kb"]),
        ("calculate 3 plus 5", ["calculator"]),
        ("compute 10 times 2", ["calculator"]),
        ("search capital france and calculate 1 plus 1", ["search_kb", "calculator"]),
        ("math: what is 8 times 7", ["calculator"]),
        ("search: what is capital of france", ["search_kb"]),
    ]
    model = DistilledPlanner()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(steps):
        loss = torch.tensor(0.0)
        for q, tools in data:
            logits = model(tokenize_question(q))
            target = tool_multihot(tools)
            loss = loss + F.binary_cross_entropy_with_logits(logits, target)
        loss = loss / len(data)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    return model


def plan_from_distilled(
    model: DistilledPlanner,
    question: str,
) -> list[PlanNode]:
    """
    把蒸馏 Planner 的工具集合编成无依赖的一层 DAG（教学简化）。

    Args:
        model: 蒸馏 planner。
        question: 用户问题。

    Returns:
        plan: 节点列表。
    """
    tools = model.predict_tools(question)
    nodes: list[PlanNode] = []
    if "search_kb" in tools:
        nodes.append(PlanNode("p_search", "search_kb", {"query": question}))
    if "calculator" in tools:
        # 极简：从问题里抠一个默认式
        expr = "1+1"
        for tok in question.replace("?", " ").split():
            if any(c.isdigit() for c in tok) and any(op in tok for op in "+-*/"):
                expr = tok
                break
        if "times" in question.lower():
            expr = "8*7"
        if "plus" in question.lower() and "3" in question and "5" in question:
            expr = "3+5"
        nodes.append(PlanNode("p_calc", "calculator", {"expr": expr}))
    if not nodes:
        nodes.append(PlanNode("p_search", "search_kb", {"query": question}))
    return nodes


def demo_distilled_planner() -> None:
    """训练小 Planner，并用它驱动一次 ReWOO。"""
    torch.manual_seed(0)
    model = train_distilled_planner()
    q = "search capital of france and calculate 3 plus 5"
    pred = model.predict_tools(q)
    print("=== distilled planner ===")
    print("predict_tools:", pred)
    assert "search_kb" in pred and "calculator" in pred

    plan = plan_from_distilled(model, q)
    result = run_rewoo(q, plan, build_default_tools())
    print(result.answer)
    print("DISTILL DEMO OK")


demo_distilled_planner()


## 4. 生产级：LangGraph Plan-and-Execute + DeepSeek

流程：`Planner` 产出结构化步骤 → `Executor` 逐步调工具 → 可选 `Replanner` → `Solver` 汇总。  
需项目根目录 `.env` 中的 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any, Literal

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"


class PlanStep(BaseModel):
    """结构化计划中的一步。"""

    id: str = Field(description="step id, e.g. s1")
    tool: Literal["calculator", "search_kb"] = Field(description="tool name")
    args: dict[str, Any] = Field(default_factory=dict, description="tool kwargs")
    depends_on: list[str] = Field(default_factory=list, description="upstream step ids")


class Plan(BaseModel):
    """Planner 输出。"""

    steps: list[PlanStep]


class ReplanDecision(BaseModel):
    """Replanner 输出：继续原计划、改计划、或结束去 Solver。"""

    action: Literal["continue", "replan", "finish"]
    steps: list[PlanStep] = Field(default_factory=list)
    reason: str = ""


class PlanExecuteState(TypedDict, total=False):
    """LangGraph 状态。"""

    question: str
    plan: list[dict[str, object]]
    evidence: dict[str, str]
    replan_count: int
    max_replans: int
    replan_action: str
    replan_reason: str
    answer: str


def get_deepseek_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model（关闭 thinking，便于结构化输出）。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def make_tool_registry() -> ToolRegistry:
    """
    Returns:
        registry: 与玩具相同的工具面（不依赖 demo cell）。
    """
    reg = ToolRegistry()
    reg.register("calculator", calculator)
    reg.register("search_kb", search_kb)
    return reg


def node_plan_to_plan_node(step: dict[str, Any]) -> PlanNode:
    """
    Args:
        step: ``PlanStep.model_dump()`` 字典。

    Returns:
        node: 玩具 DAG 节点。
    """
    return PlanNode(
        id=str(step["id"]),
        tool=str(step["tool"]),
        args=dict(step.get("args") or {}),
        depends_on=tuple(step.get("depends_on") or ()),
    )


def planner_node(state: PlanExecuteState) -> dict[str, Any]:
    """
    Args:
        state: 含 ``question``。

    Returns:
        update: ``plan`` / ``evidence`` / ``replan_count`` 初始化。
    """
    llm = get_deepseek_llm().with_structured_output(Plan)
    question = state["question"]
    plan: Plan = llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are a Planner for ReWOO / Plan-and-Execute. "
                    "Emit a small DAG of tool calls. Tools: "
                    "calculator(expr: str), search_kb(query: str). "
                    "Use depends_on for ordering. Prefer parallel independent steps. "
                    "For arithmetic use calculator; for facts use search_kb."
                )
            ),
            HumanMessage(content=question),
        ]
    )
    steps = [s.model_dump() for s in plan.steps]
    if not steps:
        # 结构化输出偶发空计划时给可执行兜底，避免 evidence 一直为空
        steps = [
            {
                "id": "s1",
                "tool": "search_kb",
                "args": {"query": question},
                "depends_on": [],
            }
        ]
    return {
        "plan": steps,
        "evidence": {},
        "replan_count": 0,
        "max_replans": state.get("max_replans", 1),
    }


def executor_node(state: PlanExecuteState) -> dict[str, Any]:
    """
    按当前 ``plan`` 拓扑执行全部 worker，写回 ``evidence``。

    Args:
        state: 含 ``plan`` / ``evidence``。

    Returns:
        update: 更新后的 ``evidence``（``node_id → 工具输出字符串``）。
    """
    plan = state.get("plan") or []
    if not plan:
        return {"evidence": dict(state.get("evidence") or {})}

    tools = make_tool_registry()
    nodes = [node_plan_to_plan_node(s) for s in plan]
    fresh = run_workers(nodes, tools)
    return {"evidence": {k: v.content for k, v in fresh.items()}}


def replanner_node(state: PlanExecuteState) -> dict[str, Any]:
    """
    根据证据决定 continue / replan / finish。

    Args:
        state: 当前状态。

    Returns:
        update: 可能含新 ``plan`` 与 ``replan_count``。
    """
    llm = get_deepseek_llm().with_structured_output(ReplanDecision)
    decision: ReplanDecision = llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are a Replanner. Given question, plan, and evidence, "
                    "choose action=finish if evidence is enough, "
                    "replan with new steps if critical failures, "
                    "else continue. Keep steps using only calculator/search_kb."
                )
            ),
            HumanMessage(
                content=json.dumps(
                    {
                        "question": state["question"],
                        "plan": state.get("plan", []),
                        "evidence": state.get("evidence", {}),
                        "replan_count": state.get("replan_count", 0),
                    },
                    ensure_ascii=False,
                )
            ),
        ]
    )
    max_replans = int(state.get("max_replans", 1))
    count = int(state.get("replan_count", 0))
    action = decision.action

    # 超过预算仍要求 replan → 强制 finish，保留已有 evidence（避免被清空后无法再执行）
    if action == "replan" and decision.steps:
        if count >= max_replans:
            return {
                "replan_action": "finish",
                "replan_reason": (
                    f"max_replans={max_replans} reached; "
                    f"keep evidence. model wanted: {decision.reason}"
                ),
            }
        return {
            "replan_action": "replan",
            "replan_reason": decision.reason,
            "plan": [s.model_dump() for s in decision.steps],
            "replan_count": count + 1,
            "evidence": {},
        }

    return {
        "replan_action": action if action in ("continue", "finish") else "finish",
        "replan_reason": decision.reason,
    }


def solver_node(state: PlanExecuteState) -> dict[str, Any]:
    """
    Args:
        state: 含问题、计划、证据。

    Returns:
        update: ``answer``。
    """
    llm = get_deepseek_llm()
    msg = llm.invoke(
        [
            SystemMessage(
                content=(
                    "You are the Solver. Using the plan and evidence only, "
                    "write a short final answer in Chinese. "
                    "If some evidence failed, degrade gracefully."
                )
            ),
            HumanMessage(
                content=json.dumps(
                    {
                        "question": state["question"],
                        "plan": state.get("plan", []),
                        "evidence": state.get("evidence", {}),
                    },
                    ensure_ascii=False,
                )
            ),
        ]
    )
    content = msg.content if isinstance(msg.content, str) else str(msg.content)
    return {"answer": content}


def route_after_replan(state: PlanExecuteState) -> str:
    """
    Args:
        state: 可能含 ``_replan_action`` / ``replan_count``。

    Returns:
        next: ``executor`` | ``solver``。
    """
    action = state.get("replan_action", "finish")
    # replan 时 evidence 已被清空且 count 已自增；只要仍标 replan 就再跑 executor
    if action == "replan":
        return "executor"
    if action == "continue":
        pending = [
            s["id"]
            for s in state.get("plan", [])
            if s["id"] not in state.get("evidence", {})
        ]
        if pending:
            return "executor"
    return "solver"


def build_plan_execute_graph() -> Any:
    """
    Returns:
        app: 编译后的 LangGraph 应用。
    """
    g: StateGraph = StateGraph(PlanExecuteState)
    g.add_node("planner", planner_node)
    g.add_node("executor", executor_node)
    g.add_node("replanner", replanner_node)
    g.add_node("solver", solver_node)
    g.add_edge(START, "planner")
    g.add_edge("planner", "executor")
    g.add_edge("executor", "replanner")
    g.add_conditional_edges(
        "replanner",
        route_after_replan,
        {"executor": "executor", "solver": "solver"},
    )
    g.add_edge("solver", END)
    return g.compile()


def run_plan_and_execute(question: str, max_replans: int = 1) -> PlanExecuteState:
    """
    Args:
        question: 用户问题。
        max_replans: 最多重新规划次数。

    Returns:
        state: 最终图状态（含 ``answer``）。
    """
    app = build_plan_execute_graph()
    return app.invoke(
        {
            "question": question,
            "max_replans": max_replans,
            "replan_count": 0,
            "plan": [],
            "evidence": {},
        }
    )


print("LangGraph Plan-and-Execute ready |", MODEL)


## 5. 生产示例


In [ ]:
def demo_plan_and_execute() -> None:
    """需要网络与 ``DEEPSEEK_API_KEY``。"""
    question = "法国的首都是什么？另外计算 (3+5)*7。"
    out = run_plan_and_execute(question, max_replans=1)
    print("=== plan ===")
    print(json.dumps(out.get("plan", []), ensure_ascii=False, indent=2))
    print("=== evidence ===")
    evidence = out.get("evidence") or {}
    print(json.dumps(evidence, ensure_ascii=False, indent=2))
    if not evidence:
        print(
            "WARN: evidence empty. replan_action=",
            out.get("replan_action"),
            "replan_reason=",
            out.get("replan_reason"),
            "replan_count=",
            out.get("replan_count"),
        )
    print("=== answer ===")
    print(out.get("answer", ""))
    assert out.get("answer")
    assert evidence, "evidence should not be empty after executor"


demo_plan_and_execute()
